In [ ]:
各信号線の故障候補値を比較、最も値が似通っている信号線ペア（ハミング距離最小のペア）を見つけるプログラム
※試作版

In [ ]:
import numpy as np
from scipy.spatial import distance  # ハミング距離を求めるためのモジュール

cir = 's344'  # 対象回路
correct_data = cir + 'output'  # 正解データファイル
net_list = 'c' + cir   # ネットリストファイル
simular_output = cir + 'simular_output'  # 並べ替え後の正解データを記述するファイル

integration = 2 # 故障候補値の統合数

# 正解データファイルを開いてデータを読み込む
with open(correct_data) as f:
  #_にいれた各文字列の空白文字を削除
  lines = [_.replace(",", "").replace("\n", "") for _ in f.readlines()]  # 各行の空白文字と改行文字を削除

# ネットリストを開いて、入力信号線数、出力信号線数、その他の信号線数を取得
with open('c' + cir, 'r') as f:
  line_inf = f.readline()  # 1行目はネットリストの情報
  output_line_num = int(line_inf.split()[1])
  input_line_num = int(line_inf.split()[2])

print(output_line_num)
print(input_line_num)  # 入力信号線の数を出力

print(lines)  # 各行のデータを出力
# 各行の縦の列をリストに格納
signal_val = ['' for _ in range(len(lines[0]))]   # 信号線の数分のリストを作成
for i in range(len(lines[0])):
  for j in range(len(lines)):
    signal_val[i] = signal_val[i] + lines[j][i]
# print(signal_val[0])
# print(signal_val[1])
# print(signal_val[2])
# print(signal_val[3])
# print(signal_val[334])
signal_num = len(signal_val)  # 信号線の総数を取得

# 探索する信号線の信号線番号のリストを作成 = 0から信号線の数までのリスト = [0, 1, 2, ..., signal_num-1] ⇒　最適なペアを探すときに、同じ信号線を比較することがないようにするのに使う
line_num_list = [0 for _ in range(signal_num)]  # 信号線の数分のリストを作成
# 診断対象となる信号線は、出力信号線以外の信号線であり、信号線番号は入力信号線、出力信号線、その他の信号線の順に1, 2, ..., となっている。そのため、入力信号線番号の後は、出力信号線の数を足している
for i in range(signal_num):
  if i < input_line_num:
    line_num_list[i] = i + 1  # 入力信号線のインデックスをそのまま格納
  else:
    line_num_list[i] = i + 1 + output_line_num
# 探索する信号線のインデックスのリスト
print(line_num_list)
print(len(line_num_list))  # 信号線の数を出力

signal_val_index_list = [i for i in range(signal_num)]  # 信号線の数分のリストを作成
print(signal_val_index_list)  # 信号線のインデックスのリストを出力
print(len(signal_val_index_list))  # 信号線の数を出力

# ハミング距離を求め,ハミング距離が最小の信号線ペアのインデックスをファイルに書き込む
f = open('hamming_distance', 'w')
f.write(f"インデックス➀ インデックス➁ ハミング距離\n")

# ハミング距離を求め,ハミング距離が最小の信号線ペアを見つける
best_pair = []  # 最適なペアを格納するリスト
best_pair_index = []  # 最適なペアのインデックスを格納するリスト
single_line = -1  # 最適なペアがない(=信号線数が奇数)信号線のインデックスを格納する変数
while len(signal_val_index_list) > 0:  # wile文にすることで、インデックスを削除しても問題なく処理できる
    idx = signal_val_index_list[0] # 探索リストの先頭の信号線のインデックス
    min_hamming = -1 # ハミング距離の最小値を格納する変数
    min_hamming_index = 0 # ハミング距離が最小の信号線のインデックスを格納する変数

    # 探索リストに残っている信号線が1本の場合 = 信号線数が奇数本の場合
    if len(signal_val_index_list) == 1:
        print(signal_val_index_list)
        single_line_index = signal_val_index_list[0]  # 探索リストに残っている信号線のインデックス
        single_line = line_num_list[idx]  # 探索リストに残っている信号線のインデックス
        print(f"信号線{single_line}には最適なペアがありません。")
        f.write(f"{single_line}\n")
        break

    for j in signal_val_index_list[1:]:  # 探索リストの先頭の信号線と他の信号線を比較
        #　文字列をリストに変換
        signal_val[idx] = list(signal_val[idx])
        signal_val[j] = list(signal_val[j])
        hamming = distance.hamming(signal_val[idx], signal_val[j]) * len(signal_val[idx])  # ハミング距離を求める
        if min_hamming == -1 or hamming < min_hamming:  # ハミング距離が最小のペアを更新
            min_hamming = hamming
            # print(j)
            min_hamming_index = j
    # ハミング距離が最小の信号線ペアを出力
    # print(f"インデックス{line_num_list[idx]}とインデックス{min_hamming_index}が最適なペアです。")
    # print(f"ハミング距離は{min_hamming}です。")
    # ハミング距離が最小のペアとハミング距離をファイルに書き込む
    f.write(f"{line_num_list[idx]} {line_num_list[min_hamming_index]} {min_hamming}\n")
    best_pair.append([line_num_list[idx], line_num_list[min_hamming_index]])  # 最適なペアをリストに追加
    best_pair_index.append([idx, min_hamming_index])  # 最適なペアのインデックスをリストに追加
    signal_val_index_list.remove(idx)  # 既に最適なペアになった信号線は探索リストから削除
    # if idx == 1:
    #    print(f"インデックス{line_num_list[idx]}とインデックス{min_hamming_index}が最適なペアです。")
    signal_val_index_list.remove(min_hamming_index)  # 既に最適なペアになった信号線は探索リストから削除

f.close()
# 探索リストに残っている信号線が1本の場合 = 信号線数が奇数本の場合
if len(signal_val_index_list) == 1:
    single_line = line_num_list[idx]  # 探索リストに残っている信号線のインデックス
    print(f"信号線{single_line}には最適なペアがありません。")

print(len(best_pair))
print("最適なペアのリスト：" + str(best_pair))  # 最適なペアを出力

# ハミング距離が最小の信号線ペアをファイルに書き込む。故障診断（diagnosis.ipynb)で使用する
with open(cir + 'pair_list', 'w') as f:
  f.write(f"統合数 {integration}\n")
  f.write(f"信号線数 {signal_num}\n")
  for i in range(len(best_pair)):  # 最適なペアの数だけループ
    f.write(str(best_pair[i][0]) + " " + str(best_pair[i][1]) + "\n")
  if len(signal_val_index_list) == 1:  # 信号線数が奇数本の場合.インデックスリストに残っている信号線のインデックスを出力
    f.write(str(single_line) + "\n")
  
print(len(best_pair))  # 最適なペアの数を出力
 
# ハミング距離が最小の信号線ペア順に故障候補値をファイルに書き込む
# 1行目は、並べ替え後の信号線番号を列挙。2行目以降は、各信号線の故障候補値を列挙
# 並べ替え前の正解データファイル（s~output）では、各行がテストパターン、各列が信号線を表していたが、並べ替え後のファイル（s~simular_output）では、各行が信号線、各列がテストパターンを表すようにする
integration_list = list(range(0, integration))  # for文の要素として使うため、リストに変換
with open(simular_output, "w") as f:
  # 最適なペアが存在しない信号線があるかどうかを示す, 0:存在しない, 1:存在する
  if len(signal_val_index_list) == 0: # ペアが存在しない信号線がない場合＝統合に余りがない場合
    f.write(f"0 #ペアが存在しない信号線はありません(最後の行の値も統合されています）\n")  # 信号線数を出力
  else:  # ペアが存在しない信号線があります＝統合に余りがある場合
    f.write(f"1 #ペアが存在しない信号線はあります（最後の行の値は統合されていません）\n") 

  # 2行目に並べ替え後の信号線番号を列挙
  for i in range(len(best_pair)):  # 最適なペアの数だけループ
    for j in integration_list:
      if (i != len(best_pair) - 1) or (j != integration - 1):
        f.write(f"{best_pair[i][j]},")
  if len(signal_val_index_list) == 0:  # ペアが存在しない信号線がない場合
    f.write(f"{best_pair[i][j]}\n")    # 最後の信号線の番号の後に改行を入れる
  else:
    f.write(f"{best_pair[i][j]},")
    f.write(f"{single_line}\n")

  # 3行目以降に各信号線の故障候補値を列挙
  for i in range(len(best_pair)):  
    for j in integration_list:
      for k in range(len(signal_val[j])):
        if k != len(signal_val[j]) - 1:
          f.write(f"{signal_val[best_pair_index[i][j]][k]},")
        else:
          f.write(f"{signal_val[best_pair_index[i][j]][k]}\n")  # 最後の信号線の値の後に改行を入れる
  
  # 信号線数が奇数本の場合、最後の行にペアがいない信号線の故障候補値を列挙
  if len(signal_val_index_list) == 1:
    for i in range(len(signal_val[single_line_index])):
        if i != len(signal_val[single_line_index]) - 1:
          f.write(f"{signal_val[single_line_index][i]},")
        else:
          f.write(f"{signal_val[single_line_index][i]}\n")
    print("信号線の本数は奇数でした。")

In [ ]:
import numpy as np
from scipy.spatial import distance  # ハミング距離を求めるためのモジュール

cir = 's344'  # 対象回路
correct_data = cir + 'output'  # 正解データファイル
net_list = 'c' + cir   # ネットリストファイル
simular_output = cir + 'simular_output'  # 並べ替え後の正解データを記述するファイル

integration = 2 # 故障候補値の統合数

# 正解データファイルを開いてデータを読み込む
with open(correct_data) as f:
  #_にいれた各文字列の空白文字を削除
  lines = [_.replace(",", "").replace("\n", "") for _ in f.readlines()]  # 各行の空白文字と改行文字を削除

# ネットリストを開いて、入力信号線数、出力信号線数、その他の信号線数を取得
with open('c' + cir, 'r') as f:
  line_inf = f.readline()  # 1行目はネットリストの情報
  output_line_num = int(line_inf.split()[1])
  input_line_num = int(line_inf.split()[2])

print(output_line_num)
print(input_line_num)  # 入力信号線の数を出力

print(lines)  # 各行のデータを出力
# 各行の縦の列をリストに格納
signal_val = ['' for _ in range(len(lines[0]))]   # 信号線の数分のリストを作成
for i in range(len(lines[0])):
  for j in range(len(lines)):
    signal_val[i] = signal_val[i] + lines[j][i]
# print(signal_val[0])
# print(signal_val[1])
# print(signal_val[2])
# print(signal_val[3])
# print(signal_val[334])
signal_num = len(signal_val)  # 信号線の総数を取得

# 探索する信号線のインデックスのリストを作成 = 0から信号線の数までのリスト = [0, 1, 2, ..., signal_num-1] ⇒　最適なペアを探すときに、同じ信号線を比較することがないようにするのに使う
index_list = [0 for _ in range(signal_num)]  # 信号線の数分のリストを作成
# 診断対象となる信号線は、出力信号線以外の信号線であり、信号線番号は入力信号線、出力信号線、その他の信号線の順に1, 2, ..., となっている。そのため、入力信号線番号の後は、出力信号線の数を足している
for i in range(signal_num):
  if i < input_line_num:
    index_list[i] = i + 1  # 入力信号線のインデックスをそのまま格納
  else:
    index_list[i] = i + 1 + output_line_num
# 探索する信号線のインデックスのリスト
print(index_list)
print(len(index_list))  # 信号線の数を出力

# ハミング距離を求め,ハミング距離が最小の信号線ペアのインデックスをファイルに書き込む
f = open('hamming_distance', 'w')
f.write(f"インデックス➀ インデックス➁ ハミング距離\n")

# ハミング距離を求め,ハミング距離が最小の信号線ペアを見つける
best_pair = []  # 最適なペアを格納するリスト
single_index = -1  # 最適なペアがない(=信号線数が奇数)信号線のインデックスを格納する変数
while len(index_list) > 1:  # wile文にすることで、インデックスを削除しても問題なく処理できる
    idx = index_list[0]  # 探索リストの先頭の信号線のインデックス
    min_hamming = 0 # ハミング距離の最小値を格納する変数
    min_hamming_index = 0 # ハミング距離が最小の信号線のインデックスを格納する変数

    # 探索リストに残っている信号線が1本の場合 = 信号線数が奇数本の場合
    if len(index_list) == 1:
        single_index = index_list[0]  # 探索リストに残っている信号線のインデックス
        print(f"インデックス{single_index}は最適なペアがありません。")
        break

    for j in index_list[1:]:
        #　文字列をリストに変換
        signal_val[i] = list(signal_val[i])
        signal_val[j] = list(signal_val[j])
        hamming = distance.hamming(signal_val[i], signal_val[j]) * len(signal_val[i])  # ハミング距離を求める
        if min_hamming == 0 or hamming < min_hamming:  # ハミング距離が最小のペアを更新
            min_hamming = hamming
            min_hamming_index = j
    # ハミング距離が最小の信号線ペアを出力
    # print(f"インデックス{i}とインデックス{min_hamming_index}が最適なペアです。")
    # print(f"ハミング距離は{min_hamming}です。")
    # ハミング距離が最小のペアとハミング距離をファイルに書き込む
    f.write(f"{i} {min_hamming_index} {min_hamming}\n")
    best_pair.append([i, min_hamming_index])  # 最適なペアをリストに追加
    index_list.remove(i)  # 既に最適なペアになった信号線は探索リストから削除
    # if i == 1:
    #    print(f"インデックス{i}とインデックス{min_hamming_index}が最適なペアです。")
    index_list.remove(min_hamming_index)  # 既に最適なペアになった信号線は探索リストから削除

f.close()
# 探索リストに残っている信号線が1本の場合 = 信号線数が奇数本の場合
if len(index_list) == 1:
    single_index = index_list[0]  # 探索リストに残っている信号線のインデックス
    print(f"インデックス{single_index}は最適なペアがありません。")

print(len(best_pair))
print("最適なペアのリスト：" + str(best_pair))  # 最適なペアを出力

# ハミング距離が最小の信号線ペアをファイルに書き込む。故障診断（diagnosis.ipynb)で使用する
with open(cir + 'pair_list', 'w') as f:
  f.write(f"統合数 {integration}\n")
  f.write(f"信号線数 {signal_num}\n")
  for i in range(len(best_pair)):
    f.write(str(best_pair[i][0]) + " " + str(best_pair[i][1]) + "\n")
  if len(index_list) == 1:  # 信号線数が奇数本の場合.インデックスリストに残っている信号線のインデックスを出力
    f.write(str(single_index) + "\n")
 
# ハミング距離が最小の信号線ペア順に故障候補値をファイルに書き込む
# 1行目は、並べ替え後の信号線番号を列挙。2行目以降は、各信号線の故障候補値を列挙
# 並べ替え前の正解データファイル（s~output）では、各行がテストパターン、各列が信号線を表していたが、並べ替え後のファイル（s~simular_output）では、各行が信号線、各列がテストパターンを表すようにする
integration_list = list(range(0, integration))  # for文の要素として使うため、リストに変換
with open(simular_output, "w") as f:
  # 最適なペアが存在しない信号線があるかどうかを示す, 0:存在しない, 1:存在する
  if len(index_list) == 0: # ペアが存在しない信号線がない場合＝統合に余りがない場合
    f.write(f"0 #ペアが存在しない信号線はありません(最後の行の値も統合されています）\n")  # 信号線数を出力
  else:  # ペアが存在しない信号線があります＝統合に余りがある場合
    f.write(f"1 #ペアが存在しない信号線はあります（最後の行の値は統合されていません）\n") 

  # 1行目に並べ替え後の信号線番号を列挙
  for i in range(len(best_pair)):
    for j in integration_list:
      if (i != len(best_pair) - 1) or (j != integration - 1):
        f.write(f"{best_pair[i][j]},")
      else:
        f.write(f"{best_pair[i][j]}\n")    # 最後の信号線の番号の後に改行を入れる

  # 2行目以降に各信号線の故障候補値を列挙
  for i in range(len(best_pair)):  
    for j in integration_list:
      for k in range(len(signal_val[j])):
        if k != len(signal_val[j]) - 1:
          f.write(f"{signal_val[best_pair[i][j]][k]},")
        else:
          f.write(f"{signal_val[best_pair[i][j]][k]}\n")  # 最後の信号線の値の後に改行を入れる
  
  # 信号線数が奇数本の場合、最後の行にペアがいない信号線の故障候補値を列挙
  if len(index_list) == 1:
    for i in range(len(signal_val[single_index])):
        if i != len(signal_val[single_index]) - 1:
          f.write(f"{signal_val[single_index][i]},")
        else:
          f.write(f"{signal_val[single_index][i]}\n")
    print("信号線の本数は奇数でした。")

10000000000000000000000000111111111100010000001000000000000000001111000000000000000111100000000000111000000000000000000000000000000000001000011000010011110001000000000000010000000000001111000000000000000001010101100001000001000001010101101101100001101000001111000000000000000001000000000000000000000000000000000001001000001110100000000
335
264
['...']


In [38]:
from scipy.spatial import distance  # ハミング距離を求めるためのモジュール

# with open(net_list) as f:
#   lines = f.readline().split()  # 1行目のデータを読み込む
# signal_num = int(lines[0])  # 信号線の数
# output_num = int(lines[1])  # 出力信号線の数
# input_num = int(lines[2])   # 入力信号線の数

# 探索する信号線のインデックスのリストを作成 = 0から信号線の数までのリスト = [0, 1, 2, ..., signal_num-1] ⇒　最適なペアを探すときに、同じ信号線を比較することがないようにするのに使う
index_list = list(range(0, len(signal_val)))  # 探索する信号線のインデックスのリスト
# print(index_list)

# ハミング距離を求め,ハミング距離が最小の信号線ペアを見つける
best_pair = []  # 最適なペアを格納するリスト
single_index = -1  # 最適なペアがない信号線のインデックスを格納する変数
for i in index_list:
    min_hamming = 0 # ハミング距離の最小値を格納する変数
    min_hamming_index = 0 # ハミング距離が最小の信号線のインデックスを格納する変数

    # 探索リストに残っている信号線が1本の場合 = 信号線数が奇数本の場合
    if len(index_list) == 1:
        single_index = index_list[0]  # 探索リストに残っている信号線のインデックス
        print(f"インデックス{single_index}は最適なペアがありません。")
        break

    for j in index_list:
        if i < j:
            #　文字列をリストに変換
            signal_val[i] = list(signal_val[i])
            signal_val[j] = list(signal_val[j])
            hamming = distance.hamming(signal_val[i], signal_val[j]) * len(signal_val[i])  # ハミング距離を求める
            if min_hamming == 0 or hamming < min_hamming:  # ハミング距離が最小のペアを更新
                min_hamming = hamming
                min_hamming_index = j
    # ハミング距離が最小の信号線ペアを出力
    # print(f"インデックス{i}とインデックス{min_hamming_index}が最適なペアです。")
    # print(f"ハミング距離は{min_hamming}です。")
    best_pair.append([i, min_hamming_index])  # 最適なペアをリストに追加
    index_list.remove(i)  # 既に最適なペアになった信号線は探索リストから削除
    index_list.remove(min_hamming_index)  # 既に最適なペアになった信号線は探索リストから削除

print("最適なペアのリスト：" + str(best_pair))  # 最適なペアを出力
# 信号線数が奇数の場合、最適なペアがない信号線のインデックスを出力
if single_index != -1:
    print("最適なペアがない信号線のインデックス：" + single_index)  # 最適なペアがない信号線のインデックスを出力

インデックス0とインデックス186が最適なペアです。
ハミング距離は54.0です。
インデックス2とインデックス6が最適なペアです。
ハミング距離は2.0です。
インデックス4とインデックス8が最適なペアです。
ハミング距離は4.0です。
インデックス7とインデックス12が最適なペアです。
ハミング距離は6.0です。
インデックス10とインデックス325が最適なペアです。
ハミング距離は9.0です。
インデックス13とインデックス287が最適なペアです。
ハミング距離は2.0です。
インデックス15とインデックス128が最適なペアです。
ハミング距離は3.0です。
インデックス17とインデックス129が最適なペアです。
ハミング距離は3.0です。
インデックス19とインデックス271が最適なペアです。
ハミング距離は3.0です。
インデックス21とインデックス283が最適なペアです。
ハミング距離は3.0です。
インデックス23とインデックス289が最適なペアです。
ハミング距離は3.0です。
インデックス25とインデックス198が最適なペアです。
ハミング距離は17.0です。
インデックス27とインデックス109が最適なペアです。
ハミング距離は59.0です。
インデックス29とインデックス146が最適なペアです。
ハミング距離は55.0です。
インデックス31とインデックス112が最適なペアです。
ハミング距離は76.0です。
インデックス33とインデックス43が最適なペアです。
ハミング距離は68.0です。
インデックス35とインデックス117が最適なペアです。
ハミング距離は35.0です。
インデックス37とインデックス255が最適なペアです。
ハミング距離は72.0です。
インデックス39とインデックス256が最適なペアです。
ハミング距離は66.0です。
インデックス41とインデックス275が最適なペアです。
ハミング距離は15.0です。
インデックス44とインデックス55が最適なペアです。
ハミング距離は3.0です。
インデックス46とインデックス212が最適なペアです。
ハミング距離は21.0です。
インデックス48とインデックス245が最適なペアです。
ハミング距離は37.0です。
インデックス50とインデックス105が最適なペアです。
ハミング距

In [40]:
import numpy as np
from scipy.spatial import distance  # ハミング距離を求めるためのモジュール

correct_data = 's344output'

# 正解データファイルを開いてデータを読み込む
with open(correct_data) as f:
  #_にいれた各文字列の空白文字を削除
  lines = [_.replace(",", "").replace("\n", "") for _ in f.readlines()]  # 各行の空白文字と改行文字を削除

# 各行の縦の列をリストに格納
signal_val = ['' for _ in range(len(lines[0]))]   # 信号線の数分のリストを作成
for i in range(len(lines[0])):
  for j in range(len(lines)):
    signal_val[i] = signal_val[i] + lines[j][i]
  

# 各列のデータをリストに格納する
data = [list(map(float, line.split())) for line in lines]
columns = list(zip(*data))

# 各リストの値を比較して、最も値が似ているリスト番号を見つける
def find_most_similar_lists(columns):
  min_diff = float('inf')
  most_similar = (None, None)
  for i in range(len(columns)):
    for j in range(i + 1, len(columns)):
      diff = np.sum(np.abs(np.array(columns[i]) - np.array(columns[j])))
      if diff < min_diff:
        min_diff = diff
        most_similar = (i, j)
  return most_similar

most_similar_lists = find_most_similar_lists(columns)
print(f"最も値が似ているリスト番号: {most_similar_lists}")

最も値が似ているリスト番号: (None, None)


In [8]:
import numpy as np
from scipy.spatial import distance  # ハミング距離を求めるためのモジュール

cir = 's344'  # 対象回路
correct_data = cir + 'output'  # 正解データファイル
net_list = 'c' + cir   # ネットリストファイル

# 正解データファイルを開いてデータを読み込む
with open(correct_data) as f:
  #_にいれた各文字列の空白文字を削除
  lines = [_.replace(",", "").replace("\n", "") for _ in f.readlines()]  # 各行の空白文字と改行文字を削除

# 各行の縦の列をリストに格納
signal_val = ['' for _ in range(len(lines[0]))]   # 信号線の数分のリストを作成
for i in range(len(lines[0])):
  for j in range(len(lines)):
    signal_val[i] = signal_val[i] + lines[j][i]
  
# 探索する信号線のインデックスのリストを作成 = 0から信号線の数までのリスト = [0, 1, 2, ..., signal_num-1] ⇒　最適なペアを探すときに、同じ信号線を比較することがないようにするのに使う
index_list = list(range(0, len(signal_val)))  # 探索する信号線のインデックスのリスト
# print(index_list)

# ハミング距離を求め,ハミング距離が最小の信号線ペアのインデックスをファイルに書き込む
f = open('hamming_distance', 'w')
f.write(f"インデックス➀ インデックス➁ ハミング距離\n")

# ハミング距離を求め,ハミング距離が最小の信号線ペアを見つける
best_pair = []  # 最適なペアを格納するリスト
single_index = -1  # 最適なペアがない信号線のインデックスを格納する変数
for i in index_list:
    min_hamming = 0 # ハミング距離の最小値を格納する変数
    min_hamming_index = 0 # ハミング距離が最小の信号線のインデックスを格納する変数

    # 探索リストに残っている信号線が1本の場合 = 信号線数が奇数本の場合
    if len(index_list) == 1:
        single_index = index_list[0]  # 探索リストに残っている信号線のインデックス
        print(f"インデックス{single_index}は最適なペアがありません。")
        break

    for j in index_list:
        if i < j:
            #　文字列をリストに変換
            signal_val[i] = list(signal_val[i])
            signal_val[j] = list(signal_val[j])
            hamming = distance.hamming(signal_val[i], signal_val[j]) * len(signal_val[i])  # ハミング距離を求める
            if min_hamming == 0 or hamming < min_hamming:  # ハミング距離が最小のペアを更新
                min_hamming = hamming
                min_hamming_index = j
    # ハミング距離が最小の信号線ペアを出力
    # print(f"インデックス{i}とインデックス{min_hamming_index}が最適なペアです。")
    # print(f"ハミング距離は{min_hamming}です。")
    # ハミング距離が最小のペアとハミング距離をファイルに書き込む
    f.write(f"{i} {min_hamming_index} {min_hamming}\n")
    best_pair.append([i, min_hamming_index])  # 最適なペアをリストに追加
    index_list.remove(i)  # 既に最適なペアになった信号線は探索リストから削除
    index_list.remove(min_hamming_index)  # 既に最適なペアになった信号線は探索リストから削除

f.close()
print("最適なペアのリスト：" + str(best_pair))  # 最適なペアを出力
# 信号線数が奇数の場合、最適なペアがない信号線のインデックスを出力
if single_index != -1:
    print("最適なペアがない信号線のインデックス：" + single_index)  # 最適なペアがない信号線のインデックスを出力

最適なペアのリスト：[[0, 186], [2, 6], [4, 8], [7, 12], [10, 325], [13, 287], [15, 128], [17, 129], [19, 271], [21, 283], [23, 289], [25, 198], [27, 109], [29, 146], [31, 112], [33, 43], [35, 117], [37, 255], [39, 256], [41, 275], [44, 55], [46, 212], [48, 245], [50, 105], [52, 54], [56, 294], [58, 121], [60, 224], [62, 258], [64, 91], [66, 93], [68, 127], [70, 240], [72, 249], [74, 76], [77, 234], [79, 118], [81, 209], [83, 98], [85, 125], [87, 89], [90, 243], [94, 327], [96, 310], [99, 111], [101, 104], [103, 114], [107, 108], [113, 184], [116, 304], [120, 278], [123, 286], [126, 307], [131, 200], [133, 288], [135, 225], [137, 138], [140, 333], [142, 315], [144, 227], [147, 175], [149, 189], [151, 181], [153, 183], [155, 199], [157, 326], [159, 262], [161, 182], [163, 165], [166, 272], [168, 197], [170, 203], [172, 188], [174, 192], [177, 180], [179, 298], [187, 299], [191, 295], [194, 273], [196, 291], [202, 260], [205, 206], [208, 285], [211, 237], [214, 279], [216, 274], [218, 281], [220, 2

In [ ]:
import numpy as np
from scipy.spatial import distance  # ハミング距離を求めるためのモジュール

cir = 's344'  # 対象回路
correct_data = cir + 'output'  # 正解データファイル
net_list = 'c' + cir   # ネットリストファイル

# 正解データファイルを開いてデータを読み込む
with open(correct_data) as f:
  #_にいれた各文字列の空白文字を削除
  lines = [_.replace(",", "").replace("\n", "") for _ in f.readlines()]  # 各行の空白文字と改行文字を削除

# 各行の縦の列をリストに格納
signal_val = ['' for _ in range(len(lines[0]))]   # 信号線の数分のリストを作成
for i in range(len(lines[0])):
  for j in range(len(lines)):
    signal_val[i] = signal_val[i] + lines[j][i]
  
# 探索する信号線のインデックスのリストを作成 = 0から信号線の数までのリスト = [0, 1, 2, ..., signal_num-1] ⇒　最適なペアを探すときに、同じ信号線を比較することがないようにするのに使う
index_list = list(range(0, len(signal_val)))  # 探索する信号線のインデックスのリスト
print(index_list)

# ハミング距離を求め,ハミング距離が最小の信号線ペアのインデックスをファイルに書き込む
f = open('hamming_distance', 'w')
f.write(f"インデックス➀ インデックス➁ ハミング距離\n")

# ハミング距離を求め,ハミング距離が最小の信号線ペアを見つける
best_pair = []  # 最適なペアを格納するリスト
single_index = -1  # 最適なペアがない(=信号線数が奇数)信号線のインデックスを格納する変数
for i in index_list:
    print(str(i)+", ")
    min_hamming = 0 # ハミング距離の最小値を格納する変数
    min_hamming_index = 0 # ハミング距離が最小の信号線のインデックスを格納する変数

    # 探索リストに残っている信号線が1本の場合 = 信号線数が奇数本の場合
    if len(index_list) == 1:
        single_index = index_list[0]  # 探索リストに残っている信号線のインデックス
        print(f"インデックス{single_index}は最適なペアがありません。")
        break

    for j in index_list:
        #　文字列をリストに変換
        signal_val[i] = list(signal_val[i])
        signal_val[j] = list(signal_val[j])
        hamming = distance.hamming(signal_val[i], signal_val[j]) * len(signal_val[i])  # ハミング距離を求める
        if min_hamming == 0 or hamming < min_hamming:  # ハミング距離が最小のペアを更新
            min_hamming = hamming
            min_hamming_index = j
    # ハミング距離が最小の信号線ペアを出力
    # print(f"インデックス{i}とインデックス{min_hamming_index}が最適なペアです。")
    # print(f"ハミング距離は{min_hamming}です。")
    # ハミング距離が最小のペアとハミング距離をファイルに書き込む
    f.write(f"{i} {min_hamming_index} {min_hamming}\n")
    best_pair.append([i, min_hamming_index])  # 最適なペアをリストに追加
    index_list.remove(i)  # 既に最適なペアになった信号線は探索リストから削除
    # if i == 1:
    #    print(f"インデックス{i}とインデックス{min_hamming_index}が最適なペアです。")
    index_list.remove(min_hamming_index)  # 既に最適なペアになった信号線は探索リストから削除

f.close()
print(len(best_pair))
print("最適なペアのリスト：" + str(best_pair))  # 最適なペアを出力
# 信号線数が奇数の場合、最適なペアがない信号線のインデックスを出力
if single_index != -1:
    print("最適なペアがない信号線のインデックス：" + single_index)  # 最適なペアがない信号線のインデックスを出力

TypeError: 'list' object is not callable

In [ ]:
import numpy as np
from scipy.spatial import distance  # ハミング距離を求めるためのモジュール

cir = 's344'  # 対象回路
correct_data = cir + 'output'  # 正解データファイル
net_list = 'c' + cir   # ネットリストファイル

# 正解データファイルを開いてデータを読み込む
with open(correct_data) as f:
  #_にいれた各文字列の空白文字を削除
  lines = [_.replace(",", "").replace("\n", "") for _ in f.readlines()]  # 各行の空白文字と改行文字を削除

# 各行の縦の列をリストに格納
signal_val = ['' for _ in range(len(lines[0]))]   # 信号線の数分のリストを作成
for i in range(len(lines[0])):
  for j in range(len(lines)):
    signal_val[i] = signal_val[i] + lines[j][i]
  
# 探索する信号線のインデックスのリストを作成 = 0から信号線の数までのリスト = [0, 1, 2, ..., signal_num-1] ⇒　最適なペアを探すときに、同じ信号線を比較することがないようにするのに使う
index_list = list(range(0, len(signal_val)))  # 探索する信号線のインデックスのリスト
print(index_list)

# ハミング距離を求め,ハミング距離が最小の信号線ペアのインデックスをファイルに書き込む
f = open('hamming_distance', 'w')
f.write(f"インデックス➀ インデックス➁ ハミング距離\n")

# ハミング距離を求め,ハミング距離が最小の信号線ペアを見つける
best_pair = []  # 最適なペアを格納するリスト
single_index = -1  # 最適なペアがない(=信号線数が奇数)信号線のインデックスを格納する変数
while len(index_list) > 1:
    i = index_list[0]  # 探索リストの先頭の信号線のインデックス
    min_hamming = 0 # ハミング距離の最小値を格納する変数
    min_hamming_index = 0 # ハミング距離が最小の信号線のインデックスを格納する変数

    # 探索リストに残っている信号線が1本の場合 = 信号線数が奇数本の場合
    if len(index_list) == 1:
        single_index = index_list[0]  # 探索リストに残っている信号線のインデックス
        print(f"インデックス{single_index}は最適なペアがありません。")
        break

    for j in index_list[1:]:
        #　文字列をリストに変換
        signal_val[i] = list(signal_val[i])
        signal_val[j] = list(signal_val[j])
        hamming = distance.hamming(signal_val[i], signal_val[j]) * len(signal_val[i])  # ハミング距離を求める
        if min_hamming == 0 or hamming < min_hamming:  # ハミング距離が最小のペアを更新
            min_hamming = hamming
            min_hamming_index = j
    # ハミング距離が最小の信号線ペアを出力
    # print(f"インデックス{i}とインデックス{min_hamming_index}が最適なペアです。")
    # print(f"ハミング距離は{min_hamming}です。")
    # ハミング距離が最小のペアとハミング距離をファイルに書き込む
    f.write(f"{i} {min_hamming_index} {min_hamming}\n")
    best_pair.append([i, min_hamming_index])  # 最適なペアをリストに追加
    index_list.remove(i)  # 既に最適なペアになった信号線は探索リストから削除
    # if i == 1:
    #    print(f"インデックス{i}とインデックス{min_hamming_index}が最適なペアです。")
    index_list.remove(min_hamming_index)  # 既に最適なペアになった信号線は探索リストから削除

f.close()
# 探索リストに残っている信号線が1本の場合 = 信号線数が奇数本の場合
if len(index_list) == 1:
    single_index = index_list[0]  # 探索リストに残っている信号線のインデックス
    print(f"インデックス{single_index}は最適なペアがありません。")
print(len(best_pair))
print("最適なペアのリスト：" + str(best_pair))  # 最適なペアを出力
# 信号線数が奇数の場合、最適なペアがない信号線のインデックスを出力
if single_index != -1:
    print("最適なペアがない信号線のインデックス：" + single_index)  # 最適なペアがない信号線のインデックスを出力

TypeError: 'list' object is not callable